In [1]:
# ============================================================
# DATA PREPROCESSING PRACTICAL ASSIGNMENT
# Mall Customer Dataset
# Covers: Data Understanding, Data Quality, Cleaning,
# Transformation, Reduction, Proximity Measures
# ============================================================

import pandas as pd
import numpy as np
import time

from sklearn.preprocessing import MinMaxScaler, StandardScaler, RobustScaler, OneHotEncoder
from sklearn.decomposition import PCA
from sklearn.feature_selection import VarianceThreshold
from sklearn.metrics.pairwise import euclidean_distances, manhattan_distances, cosine_similarity
from sklearn.model_selection import train_test_split

# ------------------------------------------------------------
# TASK 1: LOAD DATASET
# ------------------------------------------------------------
# https://github.com/swapnilsaurav/Dataset/blob/master/mall_customer_preprocessing_dataset.csv
file_path = "https://raw.githubusercontent.com/swapnilsaurav/Dataset/refs/heads/master/mall_customer_preprocessing_dataset.csv"

df = pd.read_csv(file_path)

print("Dataset Shape:", df.shape)
print("\nFirst 5 Records:")
display(df.head())

print("\nColumn Names:")
print(df.columns.tolist())

print("\nDataset Info:")
df.info()

print("\nDescriptive Statistics:")
display(df.describe(include="all"))


# ------------------------------------------------------------
# TASK 2: DATA QUALITY ASSESSMENT
# ------------------------------------------------------------

print("\nMissing Values:")
missing_summary = df.isnull().sum()
display(missing_summary[missing_summary > 0])

print("\nDuplicate Records:", df.duplicated().sum())

print("\nData Types:")
display(df.dtypes)

# Identify numeric and categorical columns
numeric_cols = df.select_dtypes(include=["int64", "float64"]).columns.tolist()
categorical_cols = df.select_dtypes(include=["object"]).columns.tolist()

print("\nNumeric Columns:", numeric_cols)
print("\nCategorical Columns:", categorical_cols)

# Outlier detection using IQR
print("\nOutlier Count using IQR Method:")
outlier_summary = {}

for col in numeric_cols:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR
    outliers = df[(df[col] < lower) | (df[col] > upper)]
    outlier_summary[col] = len(outliers)

display(pd.DataFrame(outlier_summary.items(), columns=["Column", "Outlier Count"]))


# ------------------------------------------------------------
# TASK 3: DATA CLEANING
# ------------------------------------------------------------

df_clean = df.copy()

# Remove duplicate rows
df_clean = df_clean.drop_duplicates()

# Standardize text columns: trim spaces and convert to title case
for col in categorical_cols:
    df_clean[col] = df_clean[col].astype(str).str.strip()

# Replace textual missing indicators with actual NaN
df_clean.replace(["nan", "NaN", "None", "", "NULL", "null"], np.nan, inplace=True)

# Clean Gender values
gender_map = {
    "M": "Male", "Male": "Male", "male": "Male", "MALE": "Male",
    "F": "Female", "Female": "Female", "female": "Female", "FEMALE": "Female"
}
df_clean["Gender"] = df_clean["Gender"].map(gender_map)

# Clean MembershipTier values
tier_map = {
    "basic": "Basic", "Basic": "Basic", "BASIC": "Basic",
    "silver": "Silver", "Silver": "Silver", "Silvr": "Silver",
    "gold": "Gold", "GOLD": "Gold", "Gold": "Gold",
    "platinum": "Platinum", "PLATINUM": "Platinum", "Platinum": "Platinum"
}
df_clean["MembershipTier"] = df_clean["MembershipTier"].map(tier_map)

# Standardize City, Country, DeviceType, PaymentMethod, EmailProvider
standard_text_cols = ["City", "Country", "DeviceType", "PaymentMethod", "EmailProvider", "PreferredCategory"]

for col in standard_text_cols:
    df_clean[col] = df_clean[col].astype(str).str.strip().str.title()
    df_clean[col].replace("Nan", np.nan, inplace=True)

# Fix common country variants
country_map = {
    "India": "India",
    "Ind": "India",
    "In": "India",
    "Bharat": "India"
}
df_clean["Country"] = df_clean["Country"].map(country_map).fillna(df_clean["Country"])

# Convert dates into proper datetime format
df_clean["JoinDate"] = pd.to_datetime(df_clean["JoinDate"], errors="coerce", dayfirst=False)
df_clean["LastPurchaseDate"] = pd.to_datetime(df_clean["LastPurchaseDate"], errors="coerce", dayfirst=False)

# Handle invalid ages
df_clean.loc[(df_clean["Age"] < 10) | (df_clean["Age"] > 100), "Age"] = np.nan

# Handle invalid ratings
df_clean.loc[
    (df_clean["SatisfactionRating_1_5"] < 1) |
    (df_clean["SatisfactionRating_1_5"] > 5),
    "SatisfactionRating_1_5"
] = np.nan

# Handle invalid spending score
df_clean.loc[
    (df_clean["SpendingScore_1_100"] < 1) |
    (df_clean["SpendingScore_1_100"] > 100),
    "SpendingScore_1_100"
] = np.nan

# Handle negative values in numeric columns where negative is invalid
non_negative_cols = [
    "AnnualIncome_INR", "AvgBasketValue_INR", "TotalPurchases",
    "OnlinePurchases", "StorePurchases", "LoyaltyPoints"
]

for col in non_negative_cols:
    df_clean.loc[df_clean[col] < 0, col] = np.nan

# Impute numeric missing values with median
numeric_cols_clean = df_clean.select_dtypes(include=["int64", "float64"]).columns.tolist()

for col in numeric_cols_clean:
    df_clean[col] = df_clean[col].fillna(df_clean[col].median())

# Impute categorical missing values with mode
categorical_cols_clean = df_clean.select_dtypes(include=["object"]).columns.tolist()

for col in categorical_cols_clean:
    df_clean[col] = df_clean[col].fillna(df_clean[col].mode()[0])

# Impute dates with median date
for col in ["JoinDate", "LastPurchaseDate"]:
    median_date = df_clean[col].dropna().median()
    df_clean[col] = df_clean[col].fillna(median_date)

print("\nCleaned Dataset Shape:", df_clean.shape)
print("\nMissing Values After Cleaning:")
display(df_clean.isnull().sum())


# ------------------------------------------------------------
# TASK 4: OUTLIER TREATMENT
# ------------------------------------------------------------

df_outlier_treated = df_clean.copy()

# Cap outliers using IQR capping
for col in numeric_cols_clean:
    Q1 = df_outlier_treated[col].quantile(0.25)
    Q3 = df_outlier_treated[col].quantile(0.75)
    IQR = Q3 - Q1
    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR

    df_outlier_treated[col] = np.where(
        df_outlier_treated[col] < lower,
        lower,
        df_outlier_treated[col]
    )

    df_outlier_treated[col] = np.where(
        df_outlier_treated[col] > upper,
        upper,
        df_outlier_treated[col]
    )

print("\nOutlier Treatment Completed.")


# ------------------------------------------------------------
# TASK 5: FEATURE ENGINEERING
# ------------------------------------------------------------

df_featured = df_outlier_treated.copy()

# Create customer tenure in days
df_featured["CustomerTenureDays"] = (
    df_featured["LastPurchaseDate"] - df_featured["JoinDate"]
).dt.days

df_featured["CustomerTenureDays"] = df_featured["CustomerTenureDays"].abs()

# Create purchase ratio features
df_featured["OnlinePurchaseRatio"] = (
    df_featured["OnlinePurchases"] / (df_featured["TotalPurchases"] + 1)
)

df_featured["StorePurchaseRatio"] = (
    df_featured["StorePurchases"] / (df_featured["TotalPurchases"] + 1)
)

# Create value per purchase
df_featured["ValuePerPurchase"] = (
    df_featured["AvgBasketValue_INR"] / (df_featured["TotalPurchases"] + 1)
)

print("\nFeature Engineering Completed.")
display(df_featured.head())


# ------------------------------------------------------------
# TASK 6: DATA TRANSFORMATION
# ------------------------------------------------------------

# Select numeric columns for scaling
numeric_features = df_featured.select_dtypes(include=["int64", "float64"]).columns.tolist()

# Remove target variable from transformation if present
target_col = "ChurnNextMonth"

if target_col in numeric_features:
    numeric_features.remove(target_col)

# Min-Max Normalization
minmax_scaler = MinMaxScaler()
df_minmax = df_featured.copy()
df_minmax[numeric_features] = minmax_scaler.fit_transform(df_minmax[numeric_features])

print("\nMin-Max Normalization Completed.")

# Standardization
standard_scaler = StandardScaler()
df_standard = df_featured.copy()
df_standard[numeric_features] = standard_scaler.fit_transform(df_standard[numeric_features])

print("Standardization Completed.")

# Robust Scaling
robust_scaler = RobustScaler()
df_robust = df_featured.copy()
df_robust[numeric_features] = robust_scaler.fit_transform(df_robust[numeric_features])

print("Robust Scaling Completed.")

print("\nBefore Transformation:")
display(df_featured[numeric_features].head())

print("\nAfter Standardization:")
display(df_standard[numeric_features].head())


# ------------------------------------------------------------
# TASK 7: ENCODING CATEGORICAL VARIABLES
# ------------------------------------------------------------

df_encoded = df_standard.copy()

# Drop date columns and ID/name columns before modelling
drop_cols = ["CustomerID", "CustomerName", "JoinDate", "LastPurchaseDate"]

df_encoded = df_encoded.drop(columns=drop_cols, errors="ignore")

# One-hot encode categorical variables
df_encoded = pd.get_dummies(df_encoded, drop_first=True)

print("\nEncoded Dataset Shape:", df_encoded.shape)
display(df_encoded.head())


# ------------------------------------------------------------
# TASK 8: DATA REDUCTION
# ------------------------------------------------------------

# Separate features and target
X = df_encoded.drop(columns=[target_col], errors="ignore")
y = df_encoded[target_col] if target_col in df_encoded.columns else None

print("\nOriginal Feature Count:", X.shape[1])

# 8.1 Remove low variance features
selector = VarianceThreshold(threshold=0.01)
X_variance = selector.fit_transform(X)

selected_columns = X.columns[selector.get_support()]
X_reduced_variance = pd.DataFrame(X_variance, columns=selected_columns)

print("Feature Count After Variance Threshold:", X_reduced_variance.shape[1])

# 8.2 Correlation-based feature removal
corr_matrix = X_reduced_variance.corr().abs()

upper_triangle = corr_matrix.where(
    np.triu(np.ones(corr_matrix.shape), k=1).astype(bool)
)

high_corr_features = [
    column for column in upper_triangle.columns
    if any(upper_triangle[column] > 0.90)
]

X_corr_reduced = X_reduced_variance.drop(columns=high_corr_features)

print("Highly Correlated Features Removed:", len(high_corr_features))
print("Feature Count After Correlation Reduction:", X_corr_reduced.shape[1])

# 8.3 PCA
pca = PCA(n_components=0.95)
X_pca = pca.fit_transform(X_corr_reduced)

print("PCA Components Retaining 95% Variance:", X_pca.shape[1])


# ------------------------------------------------------------
# TASK 9: SAMPLING
# ------------------------------------------------------------

# Random sample of 500 records
sample_df = df_featured.sample(n=500, random_state=42)

print("\nSample Dataset Shape:", sample_df.shape)


# ------------------------------------------------------------
# TASK 10: PROXIMITY MEASURES
# ------------------------------------------------------------

# Select 20 records for proximity analysis
proximity_data = X_corr_reduced.head(20)

# Euclidean distance
euclidean_matrix = euclidean_distances(proximity_data)

# Manhattan distance
manhattan_matrix = manhattan_distances(proximity_data)

# Cosine similarity
cosine_matrix = cosine_similarity(proximity_data)

print("\nEuclidean Distance Matrix:")
display(pd.DataFrame(euclidean_matrix))

print("\nManhattan Distance Matrix:")
display(pd.DataFrame(manhattan_matrix))

print("\nCosine Similarity Matrix:")
display(pd.DataFrame(cosine_matrix))

# Find most similar pair using cosine similarity
cosine_df = pd.DataFrame(cosine_matrix)

np.fill_diagonal(cosine_df.values, -1)

most_similar_pair = np.unravel_index(
    np.argmax(cosine_df.values),
    cosine_df.shape
)

print("\nMost Similar Records Based on Cosine Similarity:")
print("Record Index Pair:", most_similar_pair)
print("Similarity Score:", cosine_df.iloc[most_similar_pair])


# ------------------------------------------------------------
# TASK 11: TRAIN-TEST SPLIT OPTIONAL
# ------------------------------------------------------------

if y is not None:
    X_train, X_test, y_train, y_test = train_test_split(
        X_corr_reduced,
        y,
        test_size=0.2,
        random_state=42,
        stratify=y
    )

    print("\nTrain Shape:", X_train.shape)
    print("Test Shape:", X_test.shape)


# ------------------------------------------------------------
# TASK 12: SAVE CLEANED DATASETS
# ------------------------------------------------------------

df_featured.to_csv("mall_customer_cleaned_featured_dataset.csv", index=False)
df_encoded.to_csv("mall_customer_encoded_dataset.csv", index=False)
X_corr_reduced.to_csv("mall_customer_reduced_features.csv", index=False)

print("\nFiles Saved Successfully:")
print("1. mall_customer_cleaned_featured_dataset.csv")
print("2. mall_customer_encoded_dataset.csv")
print("3. mall_customer_reduced_features.csv")

Dataset Shape: (2200, 25)

First 5 Records:


,CustomerID,CustomerName,Age,Gender,AnnualIncome_INR,IncomeCurrency,SpendingScore_1_100,MembershipTier,JoinDate,LastPurchaseDate,...,PreferredCategory,City,Country,EmailProvider,DeviceType,PaymentMethod,LoyaltyPoints,SatisfactionRating_1_5,CouponUsed,ChurnNextMonth
0,CUST100001,Diya Verma,42.0,Female,241279.0,NaN,57,Silver,24-Feb-2024,14/09/2024,...,Beauty,bengaluru,NaN,gmail.com,android,Credit Card,1438.0,2.0,1,0.0
1,CUST100002,Aditya Nair,52.0,male,1135484.0,Rs,61,GOLD,02/02/2025,10-Oct-2025,...,SPORTS,PUNE,INDIA,hotmail.com,NaN,NaN,865.0,1.0,1,0.0
2,CUST100003,Saanvi Kumar,27.0,M,241860.0,INR,64,Silvr,2022-05-30,2023-12-17,...,Electronic,lucknow,India,gmail.com,Mobile,UPI,2472.0,1.0,NaN,0.0
3,CUST100004,Vihaan Iyer,27.0,FEMALE,470187.0,INR,86,Silver,01-31-2024,18-Jul-2024,...,Food Court,Pune,NaN,gmail.com,android,Credit Card,1700.0,3.0,NaN,0.0
4,CUST100005,Krishna Rao,25.0,M,630873.0,INR,53,basic,2022-01-25,01-May-2022,...,Fashion,Bengaluru,India,hotmail.com,Web,CREDIT CARD,1203.0,2.0,1,0.0



Column Names:
['CustomerID', 'CustomerName', 'Age', 'Gender', 'AnnualIncome_INR', 'IncomeCurrency', 'SpendingScore_1_100', 'MembershipTier', 'JoinDate', 'LastPurchaseDate', 'VisitFrequency', 'AvgBasketValue_INR', 'TotalPurchases', 'OnlinePurchases', 'StorePurchases', 'PreferredCategory', 'City', 'Country', 'EmailProvider', 'DeviceType', 'PaymentMethod', 'LoyaltyPoints', 'SatisfactionRating_1_5', 'CouponUsed', 'ChurnNextMonth']

Dataset Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2200 entries, 0 to 2199
Data columns (total 25 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   CustomerID              2200 non-null   object 
 1   CustomerName            2200 non-null   object 
 2   Age                     2096 non-null   float64
 3   Gender                  1970 non-null   object 
 4   AnnualIncome_INR        2059 non-null   float64
 5   IncomeCurrency          1882 non-null   object 
 6   SpendingScore_1_1

,CustomerID,CustomerName,Age,Gender,AnnualIncome_INR,IncomeCurrency,SpendingScore_1_100,MembershipTier,JoinDate,LastPurchaseDate,...,PreferredCategory,City,Country,EmailProvider,DeviceType,PaymentMethod,LoyaltyPoints,SatisfactionRating_1_5,CouponUsed,ChurnNextMonth
count,2200,2200,2096.000000,1970,2.059000e+03,1882,2200.00000,1957,2200,2124,...,2200,2184,1838,1972,1946,1980,2188.000000,2170.000000,1935,2173.000000
unique,2087,556,NaN,9,NaN,4,NaN,9,1830,1781,...,44,36,5,8,8,8,NaN,NaN,8,NaN
top,CUST100340,Rahul Mehta,NaN,MALE,NaN,INR,NaN,Gold,not available,10-19-2024,...,Fashion,Chennai,Bharat,gmail.com,IOS,upi,NaN,NaN,yes,NaN
freq,4,12,NaN,240,NaN,962,NaN,247,11,5,...,69,86,392,277,259,269,NaN,NaN,249,NaN
mean,NaN,NaN,34.781966,NaN,1.706644e+06,NaN,53.90000,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,8949.128428,3.137327,NaN,0.208928
std,NaN,NaN,18.054272,NaN,9.558517e+06,NaN,21.44778,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,87720.316207,1.697376,NaN,0.406636
min,NaN,NaN,-5.000000,NaN,-5.000000e+04,NaN,1.00000,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,-50.000000,0.000000,NaN,0.000000
25%,NaN,NaN,26.000000,NaN,4.423870e+05,NaN,39.00000,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,747.750000,2.000000,NaN,0.000000
50%,NaN,NaN,33.000000,NaN,6.511680e+05,NaN,54.00000,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,1199.000000,3.000000,NaN,0.000000
75%,NaN,NaN,42.000000,NaN,9.591475e+05,NaN,68.00000,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,1618.750000,4.000000,NaN,0.000000



Missing Values:


...


Manhattan Distance Matrix:


,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19
0,0.000000,30.045656,21.333919,22.510258,22.346436,23.496621,26.293762,30.932559,28.313843,27.186744,25.814972,18.407734,23.977720,29.524933,26.967242,27.222609,23.273330,18.740400,24.431498,26.523313
1,30.045656,0.000000,28.480727,30.714165,26.831658,24.728498,20.839570,25.004951,32.352093,19.513456,28.962939,28.459787,18.537644,25.137160,18.555916,29.258447,20.486553,25.685288,26.852797,29.598058
2,21.333919,28.480727,0.000000,25.494199,31.077479,19.384613,23.233216,23.631093,28.735730,29.390518,26.568592,20.635237,25.570799,30.090496,26.796933,27.730295,24.495683,22.923092,25.828169,22.536399
3,22.510258,30.714165,25.494199,0.000000,25.641849,26.204264,23.572142,26.751736,26.151903,27.739391,28.993499,27.205441,21.026712,21.033508,22.880200,19.870867,29.249065,22.499017,26.864424,27.860993
4,22.346436,26.831658,31.077479,25.641849,0.000000,28.513013,25.622551,32.445022,26.570895,27.010891,32.560172,27.480370,23.992857,21.332699,30.031340,22.865007,21.044149,20.901662,30.129572,26.744800
5,23.496621,24.728498,19.384613,26.204264,28.513013,0.000000,20.729877,27.020802,28.469084,24.833857,24.530082,22.989202,23.282850,22.523793,26.780788,24.765754,16.682690,21.397383,23.687199,23.818295
6,26.293762,20.839570,23.233216,23.572142,25.622551,20.729877,0.000000,28.238364,22.702748,20.888657,22.618437,25.414899,19.124127,24.709717,20.814006,23.768800,16.155296,21.846109,25.461733,26.988184
7,30.932559,25.004951,23.631093,26.751736,32.445022,27.020802,28.238364,0.000000,26.818041,26.708402,27.746371,27.859135,24.544102,25.370496,22.652700,29.844287,23.798642,27.169045,29.539208,18.959700
8,28.313843,32.352093,28.735730,26.151903,26.570895,28.469084,22.702748,26.818041,0.000000,29.106924,21.386500,24.746448,24.784429,26.631631,29.339477,30.700918,21.898522,21.848851,21.270395,22.501801
9,27.186744,19.513456,29.390518,27.739391,27.010891,24.833857,20.888657,26.708402,29.106924,0.000000,26.753211,25.717622,25.077188,23.314209,22.581947,28.273592,21.945370,21.909036,23.078597,26.380324



Cosine Similarity Matrix:


,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19
0,1.000000,-0.077561,0.403408,0.057650,0.174927,0.083601,-0.006278,-0.136975,-0.017782,0.081613,0.055072,0.472957,-0.000767,-0.260905,-0.078628,-0.043527,-0.157613,0.202202,0.052588,0.046444
1,-0.077561,1.000000,-0.036015,0.030331,0.014888,0.315757,0.338740,0.250997,0.005260,0.260013,-0.133481,-0.015558,0.406891,0.123721,0.447245,0.083533,0.372695,0.040712,0.027682,-0.124489
2,0.403408,-0.036015,1.000000,0.015095,-0.005018,0.340807,0.240101,0.187633,0.119928,-0.132830,0.118812,0.456998,0.077761,-0.344573,-0.114567,-0.072406,0.069065,0.292025,0.174681,0.309843
3,0.057650,0.030331,0.015095,1.000000,0.232989,0.062587,0.162679,0.011716,0.009682,-0.050755,-0.033298,-0.231214,0.324614,0.425397,0.241677,0.432480,-0.227815,0.004706,0.013290,-0.240967
4,0.174927,0.014888,-0.005018,0.232989,1.000000,-0.073536,0.187981,-0.185226,0.260414,-0.050412,-0.210265,-0.124715,0.009321,0.227920,-0.136221,0.275058,0.137934,0.060864,-0.138151,-0.115007
5,0.083601,0.315757,0.340807,0.062587,-0.073536,1.000000,0.257282,0.102251,-0.007215,0.027506,0.015183,0.228825,0.137186,0.129141,0.008072,-0.008454,0.404187,0.226426,0.249584,0.114677
6,-0.006278,0.338740,0.240101,0.162679,0.187981,0.257282,1.000000,0.086091,0.363520,-0.032635,0.028345,-0.025441,0.287620,-0.015963,0.188221,0.158846,0.421279,0.030495,0.046661,-0.045840
7,-0.136975,0.250997,0.187633,0.011716,-0.185226,0.102251,0.086091,1.000000,0.228660,-0.038349,0.180938,0.023340,0.197934,0.056760,0.171759,0.085484,0.204547,0.112889,0.014094,0.350032
8,-0.017782,0.005260,0.119928,0.009682,0.260414,-0.007215,0.363520,0.228660,1.000000,-0.069739,0.341474,0.152567,0.142992,0.024212,-0.083521,-0.119053,0.349172,0.108458,0.430821,0.173717
9,0.081613,0.260013,-0.132830,-0.050755,-0.050412,0.027506,-0.032635,-0.038349,-0.069739,1.000000,-0.107310,0.089663,-0.160175,0.085431,0.187312,-0.131013,0.053860,0.076771,0.144688,-0.002365



Most Similar Records Based on Cosine Similarity:
Record Index Pair: (np.int64(10), np.int64(19))
Similarity Score: 0.5111190800949935

Train Shape: (1708, 76)
Test Shape: (428, 76)

Files Saved Successfully:
1. mall_customer_cleaned_featured_dataset.csv
2. mall_customer_encoded_dataset.csv
3. mall_customer_reduced_features.csv
